# Shaft Connections, Lab 1: Key <br>
![](images/Lab_Key_Design.png)
 <br><br>
For a long drive shaft with a $ T_{nom} = 450 Nm $ , a flange coupler (Dutch: flenskopelling) is being used to connect to the input of a gearbox. <br>
The DIN 116-A60 flange is made out of EN-GJS-450, the shafts material is E295.<br>
Since impact load is expected, a $ K_A = 1.5 $ is to be considered. <br>
- Verify if a key, DIN 6885 type A, can be used
- Specify the desired ISO-tolerance classes for both shaft and hole

In [1]:
# import statements
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt

import MechDesign.Helpers as HM

from MechDesign.Units.Units import m_, mm_, kg_, s_, N_, rpm_, W_, deg_
import MechDesign.Units.UnitMethods as UM

import MechDesign.RnM as RnM

In [3]:
SC=RnM.ShaftConnection()
# SC now hold a specific instance of a ShaftConnection.  On initialization all variables are set to their respective symbolic representation, as can bee seen from the following display statement
print('SC.p_gem now hold the following symbol:')
display(SC.p_gem)

# let's start by setting the value of SC.p_gem to the expression E12_1B
SC.p_gem = SC.E12_1B_KeyAveragePressure()  # this specific instance of SC.p_gem now hold the expression, with all of the symbols
HM.EqPrint('p_gem',SC.p_gem)  # the first part is just text converted to symbol for display purposes only
HM.EqPrint('p_average',SC.p_gem) # as can be seen, the displayed formula is identical, except the left hand
# On the expression: help can be asked using hte build in MyHelp function.  Caution: this is not implemented yet for all help files 
dummy=HM.MyHelp(SC.p_gem)  # the explenations found are the default explenations

SC.p_gem now hold the following symbol:


_p_gem

Eq(p_gem, 2*_K_lambda*_T_eq/(_d*_hprime*_lprime*_n*_phi))

Eq(p_average, 2*_K_lambda*_T_eq/(_d*_hprime*_lprime*_n*_phi))

2*_K_lambda*_T_eq/(_d*_hprime*_lprime*_n*_phi)

    T_eq : [Nm] equivalent transmitted torque
    K_lambda : [] load distribution factor method C: K_lambda = 1;  method B: {tabel 12-2c}
    lprime : [mm] effective length of the key
    d : [mm] diameter of the shaft for splines: {tabel 12-3a} or addition for weakend section due to press fit screws (multiple ring press fit)
    hprime : [mm] h' is the effective height 
    n : [] number of keys or number of teeth of a spline
    phi : [] correction factor based on number of keys


In [4]:
# next step is to start adding details 
# setting up constants, based on assignment (units are specified using a trailing '_')
SC.d = 60*mm_
SC.T_nom = 450*N_*m_
SC.K_A = 1.5

# setting up constants that are not likely to change
SC.phi = 1 # 1key
SC.n = 1 # 1 key
SC.K_lambda = 1 # method C

SC.K_t = 0.92
SC.b = 18*mm_
SC.h = 11*mm_

# setting up constants chosen more arbitrary
# guessing a first value for key length
SC.l = 80*mm_  # when using l'<=1.3*d with a shaft of 60mm and key width of 18 mm l'<= 78 and l<78+18=96mm and a DIN 116 A60 has max length of 85mm

# setting up helper functions
SC.lprime = SC.E12_1_hI_KeyEffectiveLength()
SC.hprime = SC.E12_1_hJ_KeyEffectiveHeight()
SC.T_eq = SC.E12_1_hC_DynamicLoadTorque()

# all of the elements above now hold a specific value (including a unit)
# e.g.
print('example of a variable holding a value and unit')
HM.EqPrint('SC.d',SC.d)
# meanwhile, SC.p_gem is not changed, it still is holding the symbolic expression
print('p_gem is still holding a symbolic expression')
HM.EqPrint('p_gem',SC.p_gem)
t=HM.MyHelp(SC.p_gem)

example of a variable holding a value and unit


Eq(SC.d, 60*mm_)

p_gem is still holding a symbolic expression


Eq(p_gem, 2*_K_lambda*_T_eq/(_d*_hprime*_lprime*_n*_phi))

2*_K_lambda*_T_eq/(_d*_hprime*_lprime*_n*_phi)

    T_eq : [Nm] equivalent transmitted torque
    K_lambda : [] load distribution factor method C: K_lambda = 1;  method B: {tabel 12-2c}
    lprime : [mm] effective length of the key
    d : [mm] diameter of the shaft for splines: {tabel 12-3a} or addition for weakend section due to press fit screws (multiple ring press fit)
    hprime : [mm] h' is the effective height 
    n : [] number of keys or number of teeth of a spline
    phi : [] correction factor based on number of keys


In [7]:
# in order to replace the symbols with their value counterparts we have to substitute the symbols in the expression of  p_gem with the specific values (or expressions) from a specific shaft connection instance. 
# broken down into different steps:

# 1 copy expression into a temporary expression
MyExp = SC.p_gem
HM.EqPrint('MyExpOnP_gem',MyExp)
# 2 substitute the symbols in this temporary expression with the values, symbols or expressions from a specific shaft connection (SC)
MyExp2 = HM.substitute(MyExp,SC)
HM.EqPrint('MyEx_2_pOnP_gem',MyExp2)

# intermezzo: the 'substitute' function can also be used to substitute a specific symbol with something else. 
# pairs of symbol and value have to be provided, separated with a ',' 
# e.g.: setting the diameter to 2*60 and K_lambda to 2, will cancel out. Resulting in the identical value for p_gem
# Non-specified parameters will be taken from 'SC'
# Do experiment with some other values, in order to get accustomed with this function.
MyExp3 = HM.substitute(MyExp,SC,d=2*60*mm_,K_lambda=2)  
HM.EqPrint('MyEx_3_pOnP_gem',MyExp3)

# 3 replace the expression for p_gem, in a specific shaft connection, with the new expression
SC.p_gem = MyExp2

# or short:
# SC.p_gem = MD.substitute(SC.p_gem,SC)

# as can be seen from the display statement, units are still to be simplified. 
SC.p_gem = UM.m_to_mm(SC.p_gem)
t=HM.EqPrint('p_gem',SC.p_gem)

Eq(MyExpOnP_gem, 2*_K_lambda*_T_eq/(_d*_hprime*_lprime*_n*_phi))

Eq(MyEx_2_pOnP_gem, 0.07331*N_*m_/mm_**3)

Eq(MyEx_3_pOnP_gem, 0.07331*N_*m_/mm_**3)

Eq(p_gem, 73.31*N_/mm_**2)

---
  
### Conclusion:
the average pressure is well below the strength limit of the weakest material.  The design is safe
<br><br>
***

Below a set of images can be found that could serve as inspiration to create your own graphs. 


<div>
<img src="images/KeyFigExample1.PNG" width="700"/>
<img src="images/KeyFigExample2.PNG" width="700"/>
<img src="images/KeyFigExample3.PNG" width="700"/>
</div>

For a more comprehensive analysis, the key's dimensions incrementally correlate with the shaft's diameter.  
This can be accomplished by employing:  

```
# the height and width are depending on the diameter.
SC.b = sp.Piecewise((14*mm_,SC.d<=50),(16*mm_,SC.d<=58),(18*mm_,SC.d<=65))
SC.h = sp.Piecewise((9*mm_,SC.d<=50),(10*mm_,SC.d<=58),(11*mm_,SC.d<=65))
```